# Notebook 29 — Sparse Feature Emergence at the Reset Boundary

Notebook 29 studies sparse transition behavior near the modulo-30 reset path:

```text
23 → 29 → 01
```

This notebook treats lane 29 as the final admissible residue lane before reset into lane 01.


## Goals

1. Generate rolling prime-residue lane windows.
2. Track reset-boundary features along `23 → 29 → 01`.
3. Detect sparse reset events.
4. Classify reset states.
5. Export figures, CSVs, JSON summary, Markdown report, and downloadable zip.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import zscore
import json
import zipfile


In [ ]:
cwd = Path.cwd()

candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/mod30-residue-lanes"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "notebooks").exists() or (c / "src" / "mod30_residue_lanes").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

FIG_DIR = REPO_ROOT / "figures"
RES_DIR = REPO_ROOT / "results"
REP_DIR = REPO_ROOT / "reports"

for d in [FIG_DIR, RES_DIR, REP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("FIG_DIR:", FIG_DIR)
print("RES_DIR:", RES_DIR)
print("REP_DIR:", REP_DIR)


In [ ]:
MODULUS = 30
LANES = [1, 7, 11, 13, 17, 19, 23, 29]

def prime_sieve(n):
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False

    for i in range(2, int(np.sqrt(n)) + 1):
        if sieve[i]:
            sieve[i*i::i] = False

    return np.where(sieve)[0]

primes = prime_sieve(2_000_000)

prime_residues = primes % MODULUS
prime_residues = prime_residues[np.isin(prime_residues, LANES)]

len(prime_residues)


## Rolling lane windows

In [ ]:
WINDOW = 400
STEP = 40

rows = []

for start in range(0, len(prime_residues) - WINDOW, STEP):
    block = prime_residues[start:start + WINDOW]

    counts = {}
    for lane in LANES:
        counts[f"{lane:02d}"] = int(np.sum(block == lane))

    counts["window_start"] = start
    counts["window_stop"] = start + WINDOW
    rows.append(counts)

df = pd.DataFrame(rows)

df.head()


## Reset-boundary features

In [ ]:
df["gap_23_29"] = np.abs(df["23"] - df["29"])
df["gap_29_01"] = np.abs(df["29"] - df["01"])

df["reset_pressure"] = (
    df["gap_29_01"] / (df["29"] + df["01"])
)

df["boundary_pressure"] = (
    df["gap_23_29"] / (df["23"] + df["29"])
)

df.head()


## Sparse-event detection

In [ ]:
df["z_29"] = zscore(df["29"])
df["z_gap_29_01"] = zscore(df["gap_29_01"])
df["z_reset_pressure"] = zscore(df["reset_pressure"])

SPARSE_THRESHOLD = 2.0

df["sparse_event"] = (
    (np.abs(df["z_29"]) > SPARSE_THRESHOLD)
    |
    (np.abs(df["z_gap_29_01"]) > SPARSE_THRESHOLD)
    |
    (np.abs(df["z_reset_pressure"]) > SPARSE_THRESHOLD)
).astype(int)

df["sparse_event"].value_counts()


## Reset-state classification

In [ ]:
states = []

for _, row in df.iterrows():

    if row["29"] > row["01"] + 5:
        states.append("29-leading")

    elif row["01"] > row["29"] + 5:
        states.append("01-leading")

    elif row["23"] > row["29"] + 5:
        states.append("23-leading")

    elif row["reset_pressure"] > 0.12:
        states.append("reset-gap")

    else:
        states.append("balanced")

df["state"] = states

df["state"].value_counts()


## Reset-boundary lane trajectories

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(df["23"], label="23")
plt.plot(df["29"], label="29")
plt.plot(df["01"], label="01")

plt.title("Notebook 29 — Reset Boundary Lane Counts")
plt.xlabel("Rolling window")
plt.ylabel("Prime count")

plt.legend()
plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_reset_lane_counts.png", dpi=180, bbox_inches="tight")
plt.show()


## Reset pressure

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(df["reset_pressure"])

plt.title("Notebook 29 — Reset Pressure")
plt.xlabel("Rolling window")
plt.ylabel("Pressure")

plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_reset_pressure.png", dpi=180, bbox_inches="tight")
plt.show()


## Sparse feature heatmap

In [ ]:
feature_cols = [
    "23",
    "29",
    "01",
    "gap_23_29",
    "gap_29_01",
    "reset_pressure",
    "boundary_pressure",
    "z_29",
    "z_gap_29_01",
    "z_reset_pressure",
]

X = df[feature_cols].values
X = (X - X.mean(axis=0)) / X.std(axis=0)

plt.figure(figsize=(14, 8))

plt.imshow(X.T, aspect="auto")

plt.yticks(range(len(feature_cols)), feature_cols)

plt.xlabel("Rolling window")
plt.ylabel("Feature")
plt.title("Notebook 29 — Sparse Feature Heatmap")

plt.colorbar(label="Scaled value")

plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_sparse_feature_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()


## Sparse-event timeline

In [ ]:
plt.figure(figsize=(14, 3))

event_idx = np.where(df["sparse_event"] == 1)[0]

plt.scatter(
    event_idx,
    np.ones_like(event_idx),
    s=30
)

plt.title("Notebook 29 — Sparse Event Timeline")
plt.xlabel("Rolling window")
plt.yticks([])

plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_sparse_event_timeline.png", dpi=180, bbox_inches="tight")
plt.show()


## Reset state counts

In [ ]:
state_counts = df["state"].value_counts()

plt.figure(figsize=(8, 5))

plt.bar(state_counts.index, state_counts.values)

plt.title("Notebook 29 — Reset State Counts")
plt.ylabel("Count")
plt.xticks(rotation=15)

plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_reset_state_counts.png", dpi=180, bbox_inches="tight")
plt.show()


## Reset-boundary graph

In [ ]:
corr = df[["23", "29", "01"]].corr()

positions = {
    "23": (-1, 0),
    "29": (0, 0),
    "01": (1, 0),
}

plt.figure(figsize=(8, 4))

# Draw edges with correlation-scaled width.
pairs = [("23", "29"), ("29", "01"), ("23", "01")]
for a, b in pairs:
    x0, y0 = positions[a]
    x1, y1 = positions[b]
    weight = corr.loc[a, b]
    width = 1.0 + 5.0 * abs(weight)
    plt.plot([x0, x1], [y0, y1], linewidth=width, alpha=0.45)

for node, (x, y) in positions.items():
    plt.scatter([x], [y], s=2200)
    plt.text(x, y, node, ha="center", va="center", fontsize=16)

plt.title("Notebook 29 — Reset Boundary Graph")
plt.axis("off")
plt.tight_layout()

plt.savefig(FIG_DIR / "notebook29_reset_boundary_graph.png", dpi=180, bbox_inches="tight")
plt.show()

corr


## Export outputs

In [ ]:
df.to_csv(
    RES_DIR / "notebook29_reset_feature_matrix.csv",
    index=False
)

events = df[df["sparse_event"] == 1]

events.to_csv(
    RES_DIR / "notebook29_sparse_events.csv",
    index=False
)

state_counts.to_csv(
    RES_DIR / "notebook29_reset_state_counts.csv",
    header=["count"]
)

summary = {
    "repo": "mod30-residue-lanes",
    "notebook": "29_lane_residue_29",
    "title": "Sparse Feature Emergence at the Reset Boundary",
    "num_windows": int(len(df)),
    "num_sparse_events": int(df["sparse_event"].sum()),
    "mean_reset_pressure": float(df["reset_pressure"].mean()),
    "max_reset_pressure": float(df["reset_pressure"].max()),
    "mean_boundary_pressure": float(df["boundary_pressure"].mean()),
    "max_boundary_pressure": float(df["boundary_pressure"].max()),
    "state_counts": (
        df["state"]
        .value_counts()
        .to_dict()
    ),
    "constraint_view": "Lane 29 is the final admissible residue lane before modulo-30 reset into lane 01.",
}

summary_path = RES_DIR / "notebook29_reset_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

summary


## Build Markdown report

In [ ]:
report_path = REP_DIR / "report_29_sparse_feature_reset_boundary.md"

report = f"""# Report 29 — Sparse Feature Emergence at the Reset Boundary

Notebook 29 studies sparse transition behavior near the modulo-30 reset boundary:

```text
23 → 29 → 01
```

## Generated outputs

- Feature matrix CSV: <a href="../results/notebook29_reset_feature_matrix.csv">`results/notebook29_reset_feature_matrix.csv`</a>
- Sparse events CSV: <a href="../results/notebook29_sparse_events.csv">`results/notebook29_sparse_events.csv`</a>
- Reset state counts CSV: <a href="../results/notebook29_reset_state_counts.csv">`results/notebook29_reset_state_counts.csv`</a>
- Summary JSON: <a href="../results/notebook29_reset_summary.json">`results/notebook29_reset_summary.json`</a>

- Figure: <a href="../figures/notebook29_reset_lane_counts.png">`figures/notebook29_reset_lane_counts.png`</a>
- Figure: <a href="../figures/notebook29_reset_pressure.png">`figures/notebook29_reset_pressure.png`</a>
- Figure: <a href="../figures/notebook29_sparse_feature_heatmap.png">`figures/notebook29_sparse_feature_heatmap.png`</a>
- Figure: <a href="../figures/notebook29_sparse_event_timeline.png">`figures/notebook29_sparse_event_timeline.png`</a>
- Figure: <a href="../figures/notebook29_reset_state_counts.png">`figures/notebook29_reset_state_counts.png`</a>
- Figure: <a href="../figures/notebook29_reset_boundary_graph.png">`figures/notebook29_reset_boundary_graph.png`</a>

## Summary

| Metric | Value |
|---|---:|
| Windows | {summary["num_windows"]} |
| Sparse events | {summary["num_sparse_events"]} |
| Mean reset pressure | {summary["mean_reset_pressure"]:.6f} |
| Max reset pressure | {summary["max_reset_pressure"]:.6f} |
| Mean boundary pressure | {summary["mean_boundary_pressure"]:.6f} |
| Max boundary pressure | {summary["max_boundary_pressure"]:.6f} |

## State Counts

{state_counts.to_markdown()}

## Interpretation

Lane 29 behaves as the final admissible residue state before modulo-30 reset into lane 01.

Sparse transition events emerge when:

- lane 29 separates from lane 01,
- reset pressure increases,
- local boundary imbalance forms between lanes 23 and 29.

The resulting feature system forms a sparse transition manifold near the modulo reset boundary.
"""

report_path.write_text(report)

print(report_path)
print(report[:2500])


## Optional Colab download

Run this cell in Google Colab to download generated figures, results, and reports as a zip archive.


In [ ]:
# OPTIONAL COLAB DOWNLOAD

EXPORT_NAME = "notebook29_sparse_feature_outputs.zip"
export_path = REPO_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:

    for folder in [FIG_DIR, RES_DIR, REP_DIR]:

        for path in folder.glob("notebook29_*"):
            zf.write(path, arcname=str(path.relative_to(REPO_ROOT)))

        for path in folder.glob("report_29_*"):
            zf.write(path, arcname=str(path.relative_to(REPO_ROOT)))

print(export_path)

from google.colab import files
files.download(str(export_path))
